Definizione dell'architettura

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class ConvBlock(nn.Module):
    """
    Blocco convoluzionale standard: Conv2d -> BatchNorm -> ReLU.
    L’uso di BatchNorm stabilizza l’addestramento e accelera la convergenza.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels,
                kernel_size=kernel_size, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
)
    def forward(self, x):
        return self.block(x)


class CNN(nn.Module):
    """
    CNN per classificazione di immagini su C classi.
    Architettura:
        [Conv(32) -> Conv(32) -> MaxPool] x 2 (feature extraction)
        Global Average Pooling (aggregazione spaziale)
        FC(256) -> Dropout -> FC(C) (classificazione)

    Parametri
    ----------
    in_channels : int – canali dell’immagine di ingresso (es. 3 per RGB)
    num_classes : int – numero di classi di output
    """

    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            # Blocco 1
            ConvBlock(in_channels, 32),
            ConvBlock(32, 32),
            nn.MaxPool2d(kernel_size=2, stride=2), # H/2 x W/2
            nn.Dropout2d(p=0.25),

            # Blocco 2
            ConvBlock(32, 64),
            ConvBlock(64, 64),
            nn.MaxPool2d(kernel_size=2, stride=2), # H/4 x W/4
            nn.Dropout2d(p=0.25),

            # Blocco 3
            ConvBlock(64, 128),
            ConvBlock(128, 128),
            nn.MaxPool2d(kernel_size=2, stride=2), # H/8 x W/8
        )

        # Global Average Pooling: riduce ogni feature map a un singolo valore
        self.gap = nn.AdaptiveAvgPool2d(output_size=(1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(256, num_classes)
            # CrossEntropyLoss include log-softmax internamente
        )

    def forward(self, x):
        x = self.features(x) # Estrazione delle feature
        x = self.gap(x) # Aggregazione spaziale globale
        x = self.classifier(x) # Classificazione
        return x

Preparazione dei dati con Data Augmentation

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Trasformazioni per il training: augmentation + normalizzazione
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomCrop(size=32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], # CIFAR-10
    std=[0.2023, 0.1994, 0.2010])
])

# Trasformazioni per test/validazione: solo normalizzazione (no augmentation)
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
    std=[0.2023, 0.1994, 0.2010])
])

train_dataset = datasets.CIFAR10(root="./data", train=True,
download=True, transform=train_transform)

test_dataset = datasets.CIFAR10(root="./data", train=False,
download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128,
shuffle=True, num_workers=4, pin_memory=True)

test_loader = DataLoader(test_dataset, batch_size=256,
shuffle=False, num_workers=4, pin_memory=True)

Ciclo di addestramento

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN(in_channels=3, num_classes=10).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1) # regolarizzazione soft
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        # Gradient clipping: stabilizza l’addestramento

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)

            running_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, correct / total


NUM_EPOCHS = 100
best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader,optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_cnn.pth")

if epoch % 10 == 0:
    print(f"Epoch {epoch:3d} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

print(f"\nMigliore accuratezza di validazione: {best_val_acc:.4f}")

Transfer Learning con un backbone pre-addestrato

In [ ]:
from torchvision import models

def build_transfer_model(num_classes, freeze_backbone=True):

    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Sostituzione dell’ultimo strato fully connected
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.5),
        nn.Linear(256, num_classes)
    )

    return model

# Solo i parametri del classificatore sono aggiornati (feature extraction)
transfer_model = build_transfer_model(num_classes=2, freeze_backbone=True)
optimizer_ft = optim.AdamW(
    filter(lambda p: p.requires_grad, transfer_model.parameters()),
    lr=1e-4, weight_decay=1e-2
)